# Self Query Retriever — LLM 기반 메타데이터 필터링

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **SelfQueryRetriever**가 자연어 쿼리를 검색어와 메타데이터 필터로 자동 분리하는 원리 이해하기
- `AttributeInfo`로 메타데이터 스키마를 정의하는 방법 익히기
- 날짜, 카테고리, 키워드 등 다양한 필터 조건으로 검색하기

## 사전 지식

- 메타데이터(Metadata): 문서에 첨부된 구조화된 속성값 (year, category 등)
- VectorStoreRetriever 기본 사용법

---

## 핵심 아이디어

### 일반 검색과의 차이

| 방식 | 처리 과정 |
|------|----------|
| **일반 검색** | "2020년 이후 딥러닝 논문" → 전체 텍스트로 벡터 검색 |
| **Self Query** | "2020년 이후 딥러닝 논문" → 검색어: "딥러닝 논문" + 필터: `year >= 2020` |

LLM이 자연어를 분석해 검색어와 메타데이터 필터를 자동으로 분리합니다.

> **주의**: SelfQueryRetriever는 Chroma, Pinecone 등 일부 벡터스토어만 공식 지원해요. FAISS는 지원 범위가 제한적입니다.

> **실무 팁**: `verbose=True`로 설정하면 LLM이 생성한 필터 조건을 콘솔에서 확인할 수 있어서 디버깅에 유용해요.

> 🔑 **핵심 개념**: SelfQueryRetriever는 구조화된 데이터(메타데이터)와 비구조화 데이터(텍스트)를 동시에 검색합니다. 문서에 연도, 카테고리, 평점 같은 속성이 있다면 이 방식이 매우 강력해요.

> ⚠️ **자주 하는 실수**: `AttributeInfo`의 `type` 필드를 잘못 설정하면 필터가 제대로 작동하지 않습니다. 정수는 `"integer"`, 문자열은 `"string"`, 실수는 `"float"`으로 정확하게 지정하세요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
# ---------------------------------------------------
# 메타데이터가 포함된 샘플 문서를 Chroma에 저장
# ---------------------------------------------------

# ============================================================
# TODO: category, year, topic 메타데이터를 가진 5개 문서를 Chroma에 저장하세요
# 힌트: Document(page_content=..., metadata={"category": ..., "year": ..., "topic": ...})
# 예상 결과: 문서 5개 저장 완료 메시지 출력
# ============================================================

from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.schema import AttributeInfo
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

# 메타데이터가 있는 샘플 문서
docs = [
    Document(
        page_content="Scikit-learn은 2007년에 시작된 Python 기계 학습 라이브러리입니다.",
        metadata={"category": "라이브러리", "year": 2007, "topic": "기계학습"}
    ),
    Document(
        page_content="Transformer는 2017년 발표된 혁신적인 딥러닝 아키텍처입니다.",
        metadata={"category": "모델", "year": 2017, "topic": "딥러닝"}
    ),
    Document(
        page_content="Word2Vec은 2013년 Google에서 개발한 단어 임베딩 기법입니다.",
        metadata={"category": "기법", "year": 2013, "topic": "NLP"}
    ),
    Document(
        page_content="GPT-3는 2020년에 출시된 대규모 언어 모델입니다.",
        metadata={"category": "모델", "year": 2020, "topic": "언어모델"}
    ),
    Document(
        page_content="HuggingFace는 2016년 설립된 NLP 플랫폼입니다.",
        metadata={"category": "플랫폼", "year": 2016, "topic": "NLP"}
    ),
]

# Vectorstore 생성 (SelfQuery에서 공식 지원하는 Chroma 사용)
# Chroma: SelfQueryRetriever가 메타데이터 필터를 직접 전달할 수 있는 벡터스토어
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(docs, embeddings)

print(f"✅ {len(docs)}개 문서 준비 완료 (메타데이터 포함)")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ 5개 문서 준비 완료 (메타데이터 포함)


In [3]:
# ---------------------------------------------------
# AttributeInfo와 SelfQueryRetriever 생성
# ---------------------------------------------------

# ============================================================
# TODO: AttributeInfo로 category, year, topic 필드를 정의하고 SelfQueryRetriever를 생성하세요
# 힌트: AttributeInfo(name, description, type) — type은 "string" 또는 "integer"
# 예상 결과: SelfQueryRetriever 생성 완료 메시지 출력
# ============================================================

# 메타데이터 필드 정의
# AttributeInfo: LLM에게 메타데이터 필드의 의미와 타입을 알려주는 스키마
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="문서의 카테고리 (라이브러리, 모델, 기법, 플랫폼)",
        type="string",  # 문자열 필드 — eq, ne, in 연산자 지원
    ),
    AttributeInfo(
        name="year",
        description="문서가 만들어지거나 발표된 연도",
        type="integer",  # 정수 필드 — gt, lt, gte, lte 연산자 지원
    ),
    AttributeInfo(
        name="topic",
        description="주제 (기계학습, 딥러닝, NLP, 언어모델)",
        type="string",
    ),
]

# 문서 컨텐츠 설명
document_content_description = "AI 기술, 라이브러리, 모델에 대한 설명"

# LLM
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# SelfQueryRetriever 생성
# verbose=True: 생성된 구조화 쿼리를 콘솔에서 확인 가능 (디버깅에 유용)
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True
)

print("✅ SelfQueryRetriever 생성 완료")

✅ SelfQueryRetriever 생성 완료


In [4]:
# 테스트 쿼리
queries = [
    "2015년 이후에 발표된 모델을 찾아줘",
    "NLP 관련 기술을 알려줘",
    "딥러닝 모델 중에서 찾아줘"
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"📝 쿼리: {query}")
    print('='*80)
    
    try:
        docs = self_query_retriever.invoke(query)
        
        for i, doc in enumerate(docs, 1):
            print(f"\n[문서 {i}]")
            print(f"내용: {doc.page_content}")
            print(f"메타데이터: {doc.metadata}")
    except Exception as e:
        print(f"⚠️ 검색 실패: {e}")
        print("(Self Query는 LLM이 쿼리를 구조화할 수 있어야 작동합니다)")

print("\n" + "="*80)
print("\n💡 Self Query Retriever의 장점:")
print("  - 자연어 쿼리를 자동으로 구조화")
print("  - 검색어와 메타데이터 필터 자동 분리")
print("  - 더 정확한 검색 결과")



📝 쿼리: 2015년 이후에 발표된 모델을 찾아줘


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



[문서 1]
내용: HuggingFace는 2016년 설립된 NLP 플랫폼입니다.
메타데이터: {'category': '플랫폼', 'topic': 'NLP', 'year': 2016}

[문서 2]
내용: Transformer는 2017년 발표된 혁신적인 딥러닝 아키텍처입니다.
메타데이터: {'category': '모델', 'topic': '딥러닝', 'year': 2017}

[문서 3]
내용: GPT-3는 2020년에 출시된 대규모 언어 모델입니다.
메타데이터: {'category': '모델', 'topic': '언어모델', 'year': 2020}

📝 쿼리: NLP 관련 기술을 알려줘

[문서 1]
내용: HuggingFace는 2016년 설립된 NLP 플랫폼입니다.
메타데이터: {'category': '플랫폼', 'topic': 'NLP', 'year': 2016}

[문서 2]
내용: Word2Vec은 2013년 Google에서 개발한 단어 임베딩 기법입니다.
메타데이터: {'category': '기법', 'topic': 'NLP', 'year': 2013}

[문서 3]
내용: Scikit-learn은 2007년에 시작된 Python 기계 학습 라이브러리입니다.
메타데이터: {'category': '라이브러리', 'topic': '기계학습', 'year': 2007}

[문서 4]
내용: GPT-3는 2020년에 출시된 대규모 언어 모델입니다.
메타데이터: {'category': '모델', 'topic': '언어모델', 'year': 2020}

📝 쿼리: 딥러닝 모델 중에서 찾아줘

[문서 1]
내용: GPT-3는 2020년에 출시된 대규모 언어 모델입니다.
메타데이터: {'category': '모델', 'topic': '언어모델', 'year': 2020}

[문서 2]
내용: Transformer는 2017년 발표된 혁신적인 딥러닝 아키텍처입니다.
메타데이터: {'category': '모델', 'topic': '딥러닝', 'year': 20

In [5]:
# ---------------------------------------------------
# LLM이 자연어를 어떻게 구조화했는지 관찰
# ---------------------------------------------------
# query_constructor: SelfQueryRetriever 내부에서 자연어 → StructuredQuery 변환을 담당
# 이 체인을 직접 호출하면 검색어와 필터를 분리한 결과를 볼 수 있음

print(f"{'='*70}")
print("🔍 자연어 쿼리 → 구조화 쿼리 변환 관찰")
print(f"{'='*70}")

for query in queries:
    # query_constructor가 자연어를 구조화
    structured_query = self_query_retriever.query_constructor.invoke(query)

    print(f"\n📝 원본 쿼리: {query}")
    print(f"  → 검색어 (query): \"{structured_query.query}\"")
    print(f"  → 필터 (filter): {structured_query.filter}")

print(f"\n{'='*70}")
print("💡 관찰 포인트:")
print("  - LLM이 자연어에서 '검색어'와 '메타데이터 필터'를 자동 분리")
print("  - '2015년 이후' → year > 2015 필터로 변환")
print("  - 'NLP 관련' → topic = 'NLP' 필터로 변환")
print("  - 필터가 None이면 메타데이터 조건 없이 순수 벡터 검색만 수행")

🔍 자연어 쿼리 → 구조화 쿼리 변환 관찰

📝 원본 쿼리: 2015년 이후에 발표된 모델을 찾아줘
  → 검색어 (query): " "
  → 필터 (filter): comparator=<Comparator.GTE: 'gte'> attribute='year' value=2015

📝 원본 쿼리: NLP 관련 기술을 알려줘
  → 검색어 (query): "NLP"
  → 필터 (filter): None

📝 원본 쿼리: 딥러닝 모델 중에서 찾아줘
  → 검색어 (query): "딥러닝 모델"
  → 필터 (filter): None

💡 관찰 포인트:
  - LLM이 자연어에서 '검색어'와 '메타데이터 필터'를 자동 분리
  - '2015년 이후' → year > 2015 필터로 변환
  - 'NLP 관련' → topic = 'NLP' 필터로 변환
  - 필터가 None이면 메타데이터 조건 없이 순수 벡터 검색만 수행


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 자연어 → LLM이 의미 검색 쿼리 + 메타데이터 필터 자동 분리 |
| 필수 설정 | `AttributeInfo` 스키마 정의 (필드명, 타입, 설명) |
| 지원 VectorStore | Chroma, Pinecone, Weaviate 등 (FAISS 미지원) |
| 디버깅 방법 | `verbose=True` 설정 시 생성된 구조화 쿼리 확인 가능 |
| LLM 의존성 | 자연어 파싱에 LLM 호출 필요 — 응답 품질이 LLM 성능에 달림 |

### AttributeInfo 타입 지원 범위

| 데이터 타입 | 예시 | 필터 연산자 |
|-------------|------|-------------|
| `string` | 장르, 언어 | `eq`, `ne`, `in` |
| `integer` | 연도, 페이지 수 | `eq`, `gt`, `lt`, `gte`, `lte` |
| `float` | 평점, 가격 | `eq`, `gt`, `lt`, `gte`, `lte` |
| `list[string]` | 태그, 저자 목록 | `contain` |

### 다음 노트북 예고

**09-TimeWeightedVectorStoreRetriever** — 벡터 유사도와 최신성(시간 가중치)을 결합해, 오래된 문서보다 최신 문서를 우선 반환하는 검색기를 배워요.